### IMPORT LIBRARY

In [20]:
import re
import unicodedata
import pdfplumber
from collections import defaultdict
from cleantext import clean
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification

### LOAD MODEL

In [21]:
MODEL_PATH = "landing_page/backend/final_bert_model"
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForTokenClassification.from_pretrained(MODEL_PATH)
ner_pipeline = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

print(f"✓ Model loaded from {MODEL_PATH}")
print("Labels:", model.config.id2label)

coba = ner_pipeline("Saya lulusan S1 Teknik Informatika dari Universitas Indonesia dan memiliki keahlian dalam Python, Machine Learning, dan Data Analysis.")
print(coba)

Device set to use cpu


✓ Model loaded from landing_page/backend/final_bert_model
Labels: {0: 'O', 1: 'B-EDUCATION', 2: 'I-EDUCATION', 3: 'B-SKILL', 4: 'I-SKILL'}
[{'entity_group': 'EDUCATION', 'score': np.float32(1.0), 'word': 'S1 Teknik Informatika', 'start': 13, 'end': 34}, {'entity_group': 'EDUCATION', 'score': np.float32(1.0), 'word': 'Universitas Indonesia', 'start': 40, 'end': 61}, {'entity_group': 'SKILL', 'score': np.float32(0.9999999), 'word': 'Python, Machine Learning,', 'start': 90, 'end': 115}, {'entity_group': 'SKILL', 'score': np.float32(0.9999999), 'word': 'Data Analysis', 'start': 120, 'end': 133}]


### READ AND CLEAN CV

In [22]:
pdf_path = "pdf_cvs/cv-ardi.pdf"

with pdfplumber.open(pdf_path) as pdf:
    text_before = ""
    for i, page in enumerate(pdf.pages, start=1):
        page_text = page.extract_text() or ""
        text_before += f"\n--- Halaman {i} ---\n{page_text}\n"

print(f"Data CV sebelum dibersihkan:\n{text_before}")

Could get FontBBox from font descriptor because None cannot be parsed as 4 floats


Data CV sebelum dibersihkan:

--- Halaman 1 ---
.
Ardiansyah Indra Febrianto
Surabaya, East Java, Indonesia indrardi92@gmail.com 087761135290 in/ardiansyahfebrianto
SUMMARY
A passionate third-year Data Science student with strong analytical skills and a keen interest in solving real-world problems. Experienced as a data analyst with a focus on
statistics and data flow. Enthusiastic about continuous learning and skill development, actively seeking opportunities to apply data-driven solutions in practical scenarios.
EDUCATION
Applied Data Science
Politeknik Elektronika Negeri Surabaya • Surabaya, Indonesia• 2026 • 3.67 / 4.0
Natural Science Major
Sekolah Menengah Atas 4 Surabaya• Surabaya• 2022
EXPERIENCE
Digital Media Creative Team Member
Applied Data Science May 2023 - Present, Surabaya, Indonesia
• Developed and executed creative content strategies to support the branding of the Data Science program.
• Designed engaging Instagram feed posts, including fun facts about Data Science, adm

In [23]:
import pdfplumber, re, unicodedata

# --- DEFINISI STOPWORDS ---
# Tambahkan kata-kata lain di sini jika perlu
STOPWORDS = {
    "and", "of", "the", "in", "to", "for", "with", "a", "an", "is", "on", "at", "by", "as", "from",
    "dan", "yang", "di", "dengan", "untuk", "pada", "dari", "ini", "itu", "atau", "juga"
}

with pdfplumber.open(pdf_path) as pdf:
    text = "\n".join(page.extract_text() or "" for page in pdf.pages)

# 1. Normalisasi & Hapus Simbol/Bullet
text = unicodedata.normalize("NFKD", text)
text = re.sub(r"[●•▪◆■□▲▼▶◀|]", " ", text) # Ubah jadi spasi agar kata tidak dempet
text = re.sub(r"\[.*?\]", "", text)

# 2. HAPUS STOPWORDS (LOGIKA BARU)
# Memecah teks per baris, lalu per kata, filter stopword, lalu gabung lagi
processed_lines = []
for line in text.split('\n'):
    # Ambil kata-kata yang BUKAN stopword (case insensitive)
    # line.split() otomatis menghapus spasi berlebih antar kata
    words = [w for w in line.split() if w.lower() not in STOPWORDS]
    
    # Gabung kembali kata-kata tersebut menjadi kalimat
    if words:
        processed_lines.append(" ".join(words))

# Gabung kembali menjadi satu teks utuh dengan enter
text = "\n".join(processed_lines)

# 3. Format Header Otomatis (Tambah garis pemisah)
headers = r"(SUMMARY|EDUCATION|EXPERIENCE|PROJECTS|SKILLS|CERTIFICATIONS|ACHIEVEMENTS|VOLUNTEER|INVOLVEMENT|LANGUAGES|HOBBIES|REFERENCES)"
text = re.sub(f"(?i)(?m)^{headers}$", r"\n" + "="*40 + r"\n\1\n" + "="*40, text)

print(text)

Could get FontBBox from font descriptor because None cannot be parsed as 4 floats


.
Ardiansyah Indra Febrianto
Surabaya, East Java, Indonesia indrardi92@gmail.com 087761135290 in/ardiansyahfebrianto

SUMMARY
passionate third-year Data Science student strong analytical skills keen interest solving real-world problems. Experienced data analyst focus
statistics data flow. Enthusiastic about continuous learning skill development, actively seeking opportunities apply data-driven solutions practical scenarios.

EDUCATION
Applied Data Science
Politeknik Elektronika Negeri Surabaya Surabaya, Indonesia 2026 3.67 / 4.0
Natural Science Major
Sekolah Menengah Atas 4 Surabaya Surabaya 2022

EXPERIENCE
Digital Media Creative Team Member
Applied Data Science May 2023 - Present, Surabaya, Indonesia
Developed executed creative content strategies support branding Data Science program.
Designed engaging Instagram feed posts, including fun facts about Data Science, admission pathways, congratulatory messages outstanding students.
Utilized graphic design social media management skills b

### SEGMENTASI BY HEADER

In [24]:
import re

# Pastikan variabel 'text' sudah berisi teks CV yang sudah dibersihkan sebelumnya

# 1. Definisi Kata Kunci
KEYWORDS = {
    "EDUCATION": ["education", "academic", "pendidikan"],
    "EXPERIENCE": ["experience", "work", "employment", "pengalaman"],
    "SKILLS": ["skills", "technical", "tools", "keahlian"],
    "PROJECTS": ["projects", "proyek"],
    "CERTIFICATIONS": ["certifications", "sertifikasi"],
    "SUMMARY": ["summary", "profile", "about"],
    "INVOLVEMENT" : ["involvement", "activities", "organizations", "kegiatan"]
}

def parse_cv(text):
    # Inisialisasi dictionary dengan list kosong untuk setiap key
    sections = {k.lower(): [] for k in KEYWORDS}
    current = None

    for line in text.split('\n'):
        clean = line.strip()
        if not clean: continue

        # --- LOGIKA DETEKSI HEADER ---
        # Cari apakah baris ini mengandung kata kunci header DAN panjangnya kurang dari 5 kata
        header = next((k for k, v in KEYWORDS.items() if any(w in clean.lower() for w in v) and len(clean.split()) < 5), None)

        if header:
            current = header.lower()
            # PENTING: Jangan reset sections[current] = [] di sini,
            # karena jika header muncul 2x (misal beda halaman), data lama terhapus.
            continue # Skip baris ini agar judul tidak masuk ke dalam konten

        # --- LOGIKA ISI KONTEN ---
        if current:
            # PENTING: Filter garis pemisah (=== atau ---) agar tidak dianggap data
            # Regex ini mengecek jika baris isinya hanya simbol/non-huruf
            if re.match(r'^[\W_]+$', clean): 
                continue
                
            sections[current].append(clean)

    # Gabungkan list menjadi string per section
    return {k: "\n".join(v).strip() for k, v in sections.items()}

# --- Output Hasil ---
segments = parse_cv(text)

for title, content in segments.items():
    # Hanya print jika ada isinya
    if content:
        # PERBAIKAN DI SINI:
        # Variabel 'content' sudah berupa String. JANGAN di-join lagi.
        # Cukup print langsung variabel content-nya.
        print(f"[{title.upper()}]\n{content}\n" + "-"*40)

[EDUCATION]
Applied Data Science
Politeknik Elektronika Negeri Surabaya Surabaya, Indonesia 2026 3.67 / 4.0
Natural Science Major
Sekolah Menengah Atas 4 Surabaya Surabaya 2022
----------------------------------------
[EXPERIENCE]
Digital Media Creative Team Member
Applied Data Science May 2023 - Present, Surabaya, Indonesia
Developed executed creative content strategies support branding Data Science program.
Designed engaging Instagram feed posts, including fun facts about Data Science, admission pathways, congratulatory messages outstanding students.
Utilized graphic design social media management skills boost audience engagement enhance program's positive image.
Collaborated academic team ensure content aligns program's vision mission.
Basic React.js
MySkill 2024
----------------------------------------
[SKILLS]
Liaison Officer UKM ENT MBEX 2024: Coordinated managed communication between organizing committee UKM ENT, ensuring smooth
collaboration representation during event.
Member 

### NER

In [29]:
import pandas as pd
import re
import os

# ==========================================
# 1. PERSIAPAN DATABASE VALIDASI (CSV)
# ==========================================
def load_skill_database():
    valid_skills = set()
    
    # Path sesuai struktur folder di gambar Anda
    file_paths = [
        "dataset/main_data_llm/skills/skills_list.csv",
        "dataset/main_data_llm/skills/knime_skills.csv"
    ]
    
    print("Memuat database validasi skill...")
    for path in file_paths:
        if os.path.exists(path):
            try:
                # Baca CSV (asumsi header ada, ambil kolom pertama atau yang bernama 'skill')
                df = pd.read_csv(path)
                
                # Coba cari kolom yang relevan, atau ambil kolom pertama string
                cols = [c for c in df.columns if "skill" in c.lower() or "name" in c.lower()]
                target_col = cols[0] if cols else df.columns[0]
                
                # Masukkan ke set (huruf kecil) agar pencarian cepat
                skills = df[target_col].dropna().astype(str).str.lower().str.strip().tolist()
                valid_skills.update(skills)
                print(f"  - Berhasil memuat {len(skills)} skill dari {path}")
            except Exception as e:
                print(f"  - Gagal membaca {path}: {e}")
        else:
            print(f"  - File tidak ditemukan: {path}")
            
    return valid_skills

# Load skill sekali saja
VALID_SKILLS_SET = load_skill_database()

# ==========================================
# 2. CORE EXTRACTION FUNCTIONS
# ==========================================
def merge_entities_with_scores(results):
    """Menggabungkan token subword (##) menjadi kata utuh."""
    merged = []
    current_label, current_words, current_scores = None, [], []

    for e in results:
        label = e.get("entity_group", "").upper()
        raw_word = e.get("word", "")
        score = float(e.get("score", 0.0))

        if not label or not raw_word: continue

        if label == current_label:
            if raw_word.startswith("##"):
                current_words.append(raw_word.replace("##", ""))
            else:
                # Tambah spasi jika kata baru
                prefix = " " if current_words else ""
                current_words.append(prefix + raw_word)
            current_scores.append(score)
        else:
            if current_words:
                avg_score = sum(current_scores) / len(current_scores)
                merged.append({
                    "entity_group": current_label,
                    "word": "".join(current_words).strip(),
                    "score": round(avg_score, 4)
                })
            current_label = label
            current_words = [raw_word.replace("##", "")]
            current_scores = [score]

    if current_words:
        avg_score = sum(current_scores) / len(current_scores)
        merged.append({
            "entity_group": current_label,
            "word": "".join(current_words).strip(),
            "score": round(avg_score, 4)
        })
    return merged

def extract_raw_entities(text, model_pipeline, chunk_size=1024): # Chunk diperkecil sedikit agar lebih fokus
    if not text or not text.strip(): return []
    all_data = []
    # Overlap chunking untuk menghindari kata terpotong di tengah
    step = chunk_size - 100 
    chunks = [text[i:i+chunk_size] for i in range(0, len(text), step)]

    for chunk in chunks:
        raw = model_pipeline(chunk)
        cleaned = merge_entities_with_scores(raw)
        all_data.extend(cleaned)
    return all_data

# ==========================================
# 3. FUNGSI VALIDASI & SPLITTING (SOLUSI MASALAH ANDA)
# ==========================================
def validate_and_clean_skills(raw_data, valid_set):
    processed = []
    seen = set() # Untuk cegah duplikat
    
    # Kata penghubung untuk memecah kalimat panjang
    split_pattern = r'[,;/\(\)\n]| and | with | using | through | in | for | - | : '

    for item in raw_data:
        label = item['entity_group']
        word = item['word']
        score = item['score']

        # Jika EDUCATION, biarkan apa adanya (tidak perlu validasi CSV)
        if label == 'EDUCATION':
            if word not in seen:
                processed.append(item)
                seen.add(word)
            continue

        # Jika SKILL, lakukan splitting dan validasi
        if label == 'SKILL':
            # 1. Pecah kalimat panjang menjadi kandidat kata
            candidates = re.split(split_pattern, word, flags=re.IGNORECASE)
            
            for cand in candidates:
                clean_cand = cand.strip()
                # Bersihkan simbol di awal/akhir
                clean_cand = re.sub(r'^[^\w]+|[^\w]+$', '', clean_cand)
                
                if len(clean_cand) < 2: continue # Skip kata terlalu pendek (misal: "a", "I")

                lower_cand = clean_cand.lower()
                
                # 2. LOGIKA VALIDASI
                # Cek 1: Apakah kata ini ada persis di database CSV?
                if lower_cand in valid_set:
                    if clean_cand not in seen:
                        processed.append({"entity_group": "SKILL", "word": clean_cand, "score": score})
                        seen.add(clean_cand)
                
                # Cek 2 (Opsional): Jika tidak persis, apakah dia MENGANDUNG skill valid?
                # Misal extracted: "Proficient in Python" -> CSV punya "python" -> Ambil "python"
                else:
                    found_in_db = False
                    for db_skill in valid_set:
                        # Cek apakah skill db ada di dalam teks (word boundary check agar akurat)
                        # misal "Java" ketemu di "Javascript" -> Jangan ambil. "Java" di "Java Programming" -> Ambil.
                        if re.search(r'\b' + re.escape(db_skill) + r'\b', lower_cand):
                            # Kembalikan format asli (Title Case) dari database atau text
                            final_word = db_skill.title() 
                            if final_word not in seen:
                                processed.append({"entity_group": "SKILL", "word": final_word, "score": score})
                                seen.add(final_word)
                            found_in_db = True
                            # Break agar tidak double detect (misal "Ms Excel" kena "Excel" dan "Ms")
                            break 
                    
                    # Jika user ingin menampilkan 'per kata' dan database kosong, 
                    # kita bisa skip validasi strict, tapi untuk sekarang kita ikuti permintaan validasi file path.
                    pass

    return processed

def print_clean_table(data, title):
    print(f"\n=== TABEL {title.upper()} (Validated) ===")
    if not data:
        print("Tidak ada data valid ditemukan.")
        return
    
    # Sorting score tertinggi
    data = sorted(data, key=lambda x: x['score'], reverse=True)

    print(f"{'ENTITY GROUP':<15} | {'SCORE':<10} | {'WORD'}")
    print("-" * 80)
    for item in data:
        print(f"{item['entity_group']:<15} | {item['score']:<10} | {item['word']}")

# ==========================================
# 4. EKSEKUSI UTAMA
# ==========================================

print("Sedang memproses...")

# A. Siapkan Teks
text_edu = segments.get("education", "")
# Gabung semua teks selain education untuk skill search
text_skills = "\n".join([v for k, v in segments.items() if k != "education"])

# B. Ekstrak Raw (Mentah)
raw_edu = extract_raw_entities(text_edu, ner_pipeline)
raw_skills = extract_raw_entities(text_skills, ner_pipeline)

# C. Validasi & Bersihkan
# Education tidak divalidasi ke CSV skill, Skill divalidasi ke CSV
final_education = validate_and_clean_skills(raw_edu, set()) # Set kosong karena Education lewat bypass
final_skills = validate_and_clean_skills(raw_skills, VALID_SKILLS_SET)

# D. Tampilkan
print_clean_table(final_education, "Education")
print_clean_table(final_skills, "Skills (Sesuai CSV)")

Memuat database validasi skill...
  - Gagal membaca dataset/main_data_llm/skills/skills_list.csv: 'utf-8' codec can't decode byte 0x92 in position 262: invalid start byte
  - Berhasil memuat 301 skill dari dataset/main_data_llm/skills/knime_skills.csv
Sedang memproses...

=== TABEL EDUCATION (Validated) ===
ENTITY GROUP    | SCORE      | WORD
--------------------------------------------------------------------------------
EDUCATION       | 1.0        | Applied Data Science Politeknik Elektronika Negeri Surabaya Surabaya, Indonesia
EDUCATION       | 1.0        | Natural Science Major Sekolah Menengah Atas 4 Surabaya Surabaya 202

=== TABEL SKILLS (SESUAI CSV) (Validated) ===
ENTITY GROUP    | SCORE      | WORD
--------------------------------------------------------------------------------
EDUCATION       | 1.0        | Applied Data Science
EDUCATION       | 1.0        | Junior Staff Politeknik Elektronika Negeri Surabaya Surabaya External Department Lembaga Minat Bakat PENS March
EDUCA

### JOB MATCHING

In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def match_skills(cv_skills, job_skills, threshold=0.3):
    matched = []
    cv_texts = [s.lower() for s in cv_skills]
    job_texts = [s.lower() for s in job_skills]

    for job_skill in job_texts:
        # gabungkan skill CV + skill job untuk vectorizer
        corpus = cv_texts + [job_skill]
        vectorizer = TfidfVectorizer().fit(corpus)
        tfidf_matrix = vectorizer.transform(corpus)

        # similarity antara job_skill dengan semua skill CV
        sim_scores = cosine_similarity(tfidf_matrix[-1], tfidf_matrix[:-1]).flatten()

        # ambil skill CV dengan similarity tertinggi
        best_idx = sim_scores.argmax()
        best_score = sim_scores[best_idx]

        if best_score >= threshold:
            matched.append(job_skill.title())

    score = round(len(matched) / len(job_texts) * 100, 2) if job_texts else 0
    return sorted(set(matched)), score


### EXTRACTION

In [28]:
job_description = {
    "title": "Software Developer",
    "education_required": ["Bachelor's Degree in Computer Science"],
    "skills_required": ["Python", "Django", "Machine Learning", "SQL", "Spark"],
}

# Ambil hasil ekstraksi dari NER
cv_education = cv_entities.get("education", [])
cv_skills_raw = cv_entities.get("skills", [])
cv_skills = format_skills(cv_skills_raw)

# Refinement: filter skill CV agar hanya yang s dengan job requirement
cv_skills_refined = refine_skills(cv_skills_raw, job_description["skills_required"])

# Matching dengan TF-IDF + Cosine Similarity
matched_skills, match_score = match_skills(cv_skills_refined, job_description["skills_required"])

NameError: name 'cv_entities' is not defined

### RESULT

In [ ]:
print("\n" + "="*70)
print("CV EXTRACTION RESULT")
print("="*70)
print(f"Education: {cv_education}")
print(f"Skills (refined): {cv_skills_refined}")
print(f"\nJob Match Score: {match_score}%")
print("-" * 70)
print(f"Required Skills: {job_description['skills_required']}")
print(f"Matched Skills: {matched_skills}")
print(f"Match Rate: {len(matched_skills)}/{len(job_description['skills_required'])} = {match_score}%")
print("-" * 70)


CV EXTRACTION RESULT
Education: ['Ba', 'Development with PHPImplemented the backend logic usingto process user input', 'faculty', 'Applied Data Science Politeknik Elektronika Negeri Surabaya Surabaya Indonesia Natural Science Major Sekolah Menengah Atas Surabaya Surabaya Development of a MultiNode Greenhouse Condition Dashboard Based on Data Mining as an Effort to Empower Melon Farmers in Wates District Blitar Regency Republik Melon dashtaniid December Present Backend Developer Flask Supabase cPanel Developed back', 'FullStack Developer EEPIS LEN', 'A', 'HL and', 'services using Flask framework to handle API requests andIntegrated Supabase for database management ensuring seamless connection andreDeployed machine learning models and backend applications on cPanel hosting handling', 'Database Management with phpMyAdmin MySet up and managed the My', 'handling room', 'NG SPE for Campus Room Booking System Applied Data Science of PENS February June Developed Frontend with HTML and CSSDesi